# Compresión y redundancia en cadenas

Este cuaderno conecta entropía, regularidad y compresión con ejemplos pequeños.

No implementa un compresor industrial. La meta es ver por qué las cadenas con patrones pueden describirse con menos bits que cadenas sin estructura evidente.

In [ ]:
import math
from collections import Counter


def frecuencias(texto):
    total = len(texto)
    return {simbolo: cuenta / total for simbolo, cuenta in Counter(texto).items()}


def entropia(distribucion):
    return -sum(p * math.log2(p) for p in distribucion.values() if p > 0)


def bits_estimados_por_frecuencia(texto):
    return len(texto) * entropia(frecuencias(texto))

## Frecuencias y entropía empírica

Comparamos cadenas con distinta regularidad.

In [ ]:
cadenas = {
    "repetida": "0" * 64,
    "periodica": "01" * 32,
    "bloques": "0" * 40 + "1" * 24,
    "mezclada": "0100110101110010110100101110010101101001011100101011010010110010",
}

for nombre, cadena in cadenas.items():
    h = entropia(frecuencias(cadena))
    print(f"{nombre:9s} longitud = {len(cadena):2d}  H_emp = {h:.4f} bits/símbolo")

La entropía empírica basada solo en frecuencias no detecta todos los patrones. Una cadena `010101...` tiene frecuencias equilibradas, pero es muy regular.

## Codificación por longitud de rachas

Una codificación muy simple para cadenas binarias es guardar rachas consecutivas: símbolo y longitud.

In [ ]:
def rle(cadena):
    if not cadena:
        return []
    resultado = []
    actual = cadena[0]
    cuenta = 1
    for simbolo in cadena[1:]:
        if simbolo == actual:
            cuenta += 1
        else:
            resultado.append((actual, cuenta))
            actual = simbolo
            cuenta = 1
    resultado.append((actual, cuenta))
    return resultado


def reconstruir_rle(rachas):
    return "".join(simbolo * cuenta for simbolo, cuenta in rachas)


for nombre, cadena in cadenas.items():
    rachas = rle(cadena)
    print(f"{nombre:9s} rachas = {len(rachas):2d}  reconstruye = {reconstruir_rle(rachas) == cadena}")

RLE funciona bien cuando hay rachas largas, pero puede empeorar cadenas que alternan mucho.

In [ ]:
for nombre, cadena in cadenas.items():
    print(nombre)
    print("  primeros símbolos:", cadena[:32])
    print("  rachas:", rle(cadena)[:10])
    print("  bits por frecuencia, estimado:", round(bits_estimados_por_frecuencia(cadena), 2))

## Para experimentar

1. Añade una cadena con muchas rachas largas.
2. Añade una cadena perfectamente alternante y observa qué ocurre con RLE.
3. Explica por qué frecuencia y patrón no son la misma cosa.